In [ ]:
import pandas as pd

url = "https://archive.ics.uci.edu/ml/machine-learning-databases/00601/ai4i2020.csv"

df = pd.read_csv(url)

print("Shape:", df.shape)
print("\nColumns:")
print(df.columns.tolist())

print("\nFirst 5 rows:")
display(df.head())

Shape: (10000, 14)

Columns:
['UDI', 'Product ID', 'Type', 'Air temperature [K]', 'Process temperature [K]', 'Rotational speed [rpm]', 'Torque [Nm]', 'Tool wear [min]', 'Machine failure', 'TWF', 'HDF', 'PWF', 'OSF', 'RNF']

First 5 rows:


,UDI,Product ID,Type,Air temperature [K],Process temperature [K],Rotational speed [rpm],Torque [Nm],Tool wear [min],Machine failure,TWF,HDF,PWF,OSF,RNF
0,1,M14860,M,298.1,308.6,1551,42.8,0,0,0,0,0,0,0
1,2,L47181,L,298.2,308.7,1408,46.3,3,0,0,0,0,0,0
2,3,L47182,L,298.1,308.5,1498,49.4,5,0,0,0,0,0,0
3,4,L47183,L,298.2,308.6,1433,39.5,7,0,0,0,0,0,0
4,5,L47184,L,298.2,308.7,1408,40.0,9,0,0,0,0,0,0


In [ ]:
print("Missing values:")
print(df.isnull().sum())

print("\nDuplicates:", df.duplicated().sum())

print("\nData types:")
print(df.dtypes)

print("\nDataset information:")
df.info()

Missing values:
UDI                        0
Product ID                 0
Type                       0
Air temperature [K]        0
Process temperature [K]    0
Rotational speed [rpm]     0
Torque [Nm]                0
Tool wear [min]            0
Machine failure            0
TWF                        0
HDF                        0
PWF                        0
OSF                        0
RNF                        0
dtype: int64

Duplicates: 0

Data types:
UDI                          int64
Product ID                  object
Type                        object
Air temperature [K]        float64
Process temperature [K]    float64
Rotational speed [rpm]       int64
Torque [Nm]                float64
Tool wear [min]              int64
Machine failure              int64
TWF                          int64
HDF                          int64
PWF                          int64
OSF                          int64
RNF                          int64
dtype: object

Dataset information:
<class 'pan

In [ ]:
print(df["Machine failure"].value_counts())

Machine failure
0    9661
1     339
Name: count, dtype: int64


In [ ]:
failure_cols = ["TWF", "HDF", "PWF", "OSF", "RNF"]

for col in failure_cols:
    print(f"{col}: {df[col].sum()}")

TWF: 46
HDF: 115
PWF: 95
OSF: 98
RNF: 19


In [ ]:
failure_cols = ["TWF", "HDF", "PWF", "OSF", "RNF"]

df["Failure_Count"] = df[failure_cols].sum(axis=1)

print(df["Failure_Count"].value_counts())

Failure_Count
0    9652
1     324
2      23
3       1
Name: count, dtype: int64


In [ ]:
failure_cols = ["TWF", "HDF", "PWF", "OSF", "RNF"]

def get_failure_type(row):
    active = [col for col in failure_cols if row[col] == 1]

    if len(active) == 0:
        return "No Failure"
    elif len(active) == 1:
        return active[0]
    else:
        return "Multiple Failure"

df["Failure Type"] = df.apply(get_failure_type, axis=1)

print(df["Failure Type"].value_counts())

Failure Type
No Failure          9652
HDF                  106
PWF                   80
OSF                   78
TWF                   42
Multiple Failure      24
RNF                   18
Name: count, dtype: int64


In [ ]:
print(pd.crosstab(df["Machine failure"], df["Failure Type"]))

Failure Type     HDF  Multiple Failure  No Failure  OSF  PWF  RNF  TWF
Machine failure                                                       
0                  0                 0        9643    0    0   18    0
1                106                24           9   78   80    0   42


In [ ]:
# Features
features = [
    "Type",
    "Air temperature [K]",
    "Process temperature [K]",
    "Rotational speed [rpm]",
    "Torque [Nm]",
    "Tool wear [min]"
]

# Target
target = "Machine failure"

X = df[features].copy()
y = df[target].copy()

print("X shape:", X.shape)
print("y shape:", y.shape)

print("\nTarget distribution:")
print(y.value_counts())

X shape: (10000, 6)
y shape: (10000,)

Target distribution:
Machine failure
0    9661
1     339
Name: count, dtype: int64


In [ ]:
from sklearn.model_selection import train_test_split

# 1) Train = 70% ، والباقي 30%
X_train, X_temp, y_train, y_temp = train_test_split(
    X,
    y,
    test_size=0.30,
    random_state=42,
    stratify=y
)

# 2) تقسيم الـ30% إلى Validation = 15% و Test = 15%
X_val, X_test, y_val, y_test = train_test_split(
    X_temp,
    y_temp,
    test_size=0.50,
    random_state=42,
    stratify=y_temp
)

# عرض الأحجام
print("Train:", X_train.shape, y_train.shape)
print("Validation:", X_val.shape, y_val.shape)
print("Test:", X_test.shape, y_test.shape)

# توزيع الـTarget
print("\nTrain:")
print(y_train.value_counts())

print("\nValidation:")
print(y_val.value_counts())

print("\nTest:")
print(y_test.value_counts())

Train: (7000, 6) (7000,)
Validation: (1500, 6) (1500,)
Test: (1500, 6) (1500,)

Train:
Machine failure
0    6763
1     237
Name: count, dtype: int64

Validation:
Machine failure
0    1449
1      51
Name: count, dtype: int64

Test:
Machine failure
0    1449
1      51
Name: count, dtype: int64


In [ ]:
!pip install -q imbalanced-learn

In [ ]:
from sklearn.preprocessing import LabelEncoder

encoder = LabelEncoder()

X_train["Type"] = encoder.fit_transform(X_train["Type"])
X_val["Type"] = encoder.transform(X_val["Type"])
X_test["Type"] = encoder.transform(X_test["Type"])

print(encoder.classes_)

['H' 'L' 'M']


In [ ]:
print(X_train.head())
print(X_train.shape)

      Type  Air temperature [K]  Process temperature [K]  \
1888     2                297.8                    307.4   
4858     1                303.7                    312.3   
8990     1                297.2                    307.9   
4901     2                303.6                    312.3   
7957     0                300.9                    311.9   

      Rotational speed [rpm]  Torque [Nm]  Tool wear [min]  
1888                    1902         24.3              129  
4858                    1349         51.0              105  
8990                    1493         38.4              146  
4901                    1630         32.4              223  
7957                    2140         16.5               43  
(7000, 6)


In [ ]:
from imblearn.over_sampling import SMOTENC

smote = SMOTENC(
    categorical_features=[0],
    random_state=42
)

X_train_balanced, y_train_balanced = smote.fit_resample(
    X_train,
    y_train
)

print("قبل التوازن:")
print(y_train.value_counts())

print("\nبعد التوازن:")
print(y_train_balanced.value_counts())

print("\nShape:", X_train_balanced.shape)

قبل التوازن:
Machine failure
0    6763
1     237
Name: count, dtype: int64

بعد التوازن:
Machine failure
0    6763
1    6763
Name: count, dtype: int64

Shape: (13526, 6)


In [ ]:
from sklearn.ensemble import RandomForestClassifier

model1 = RandomForestClassifier(
    n_estimators=200,
    random_state=42,
    class_weight=None
)

model1.fit(X_train_balanced, y_train_balanced)

print("Model 1 trained successfully!")

Model 1 trained successfully!


In [ ]:
y_pred = model1.predict(X_test)

print(y_pred[:20])

[0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0]


In [ ]:
print("Actual failures:", y_test.sum())
print("Predicted failures:", y_pred.sum())

Actual failures: 51
Predicted failures: 91


In [ ]:
from sklearn.metrics import classification_report, confusion_matrix

print(confusion_matrix(y_test, y_pred))
print(classification_report(y_test, y_pred))

[[1391   58]
 [  18   33]]
              precision    recall  f1-score   support

           0       0.99      0.96      0.97      1449
           1       0.36      0.65      0.46        51

    accuracy                           0.95      1500
   macro avg       0.67      0.80      0.72      1500
weighted avg       0.97      0.95      0.96      1500



In [ ]:
y_prob = model1.predict_proba(X_test)[:, 1]

print(y_prob[:20])

[0.    0.    0.    0.13  0.    0.16  0.005 0.    0.    0.34  0.005 0.005
 0.    0.055 0.    0.    0.    0.005 0.    0.   ]


In [ ]:
from sklearn.metrics import roc_auc_score, average_precision_score

print("ROC-AUC:", roc_auc_score(y_test, y_prob))
print("PR-AUC:", average_precision_score(y_test, y_prob))

ROC-AUC: 0.9618533403699644
PR-AUC: 0.5971084543152967


In [ ]:
import numpy as np
from sklearn.metrics import f1_score

thresholds = np.arange(0.05, 1.00, 0.05)

for t in thresholds:
    pred = (y_prob >= t).astype(int)
    print(f"Threshold {t:.2f} → F1: {f1_score(y_test, pred):.3f}")

Threshold 0.05 → F1: 0.238
Threshold 0.10 → F1: 0.323
Threshold 0.15 → F1: 0.381
Threshold 0.20 → F1: 0.410
Threshold 0.25 → F1: 0.430
Threshold 0.30 → F1: 0.459
Threshold 0.35 → F1: 0.435
Threshold 0.40 → F1: 0.430
Threshold 0.45 → F1: 0.449
Threshold 0.50 → F1: 0.462
Threshold 0.55 → F1: 0.466
Threshold 0.60 → F1: 0.504
Threshold 0.65 → F1: 0.519
Threshold 0.70 → F1: 0.505
Threshold 0.75 → F1: 0.506
Threshold 0.80 → F1: 0.519
Threshold 0.85 → F1: 0.500
Threshold 0.90 → F1: 0.507
Threshold 0.95 → F1: 0.241


In [ ]:
from sklearn.metrics import classification_report

y_pred_065 = (y_prob >= 0.65).astype(int)

print(classification_report(y_test, y_pred_065))

              precision    recall  f1-score   support

           0       0.98      0.98      0.98      1449
           1       0.50      0.53      0.51        51

    accuracy                           0.97      1500
   macro avg       0.74      0.76      0.75      1500
weighted avg       0.97      0.97      0.97      1500



In [ ]:
from sklearn.metrics import precision_score, recall_score, f1_score

for t in np.arange(0.05, 1.00, 0.05):
    pred = (y_prob >= t).astype(int)
    recall = recall_score(y_test, pred)
    precision = precision_score(y_test, pred, zero_division=0)
    f1 = f1_score(y_test, pred)

    if recall >= 0.80:
        print(f"Threshold: {t:.2f} | Recall: {recall:.2f} | Precision: {precision:.2f} | F1: {f1:.2f}")

Threshold: 0.05 | Recall: 0.96 | Precision: 0.14 | F1: 0.24
Threshold: 0.10 | Recall: 0.94 | Precision: 0.20 | F1: 0.32
Threshold: 0.15 | Recall: 0.94 | Precision: 0.24 | F1: 0.38
Threshold: 0.20 | Recall: 0.92 | Precision: 0.26 | F1: 0.41
Threshold: 0.25 | Recall: 0.90 | Precision: 0.28 | F1: 0.43
Threshold: 0.30 | Recall: 0.88 | Precision: 0.31 | F1: 0.46


In [ ]:
from sklearn.calibration import CalibratedClassifierCV

calibrated_model1 = CalibratedClassifierCV(
    model1,
    method="sigmoid",
    cv=5
)

calibrated_model1.fit(X_train_balanced, y_train_balanced)

y_prob_calibrated = calibrated_model1.predict_proba(X_test)[:, 1]

print(y_prob_calibrated[:20])

[0.00257622 0.00255419 0.0026656  0.02063317 0.00255419 0.02269592
 0.00262241 0.00255419 0.00257239 0.17602496 0.00266356 0.00255419
 0.00255419 0.00887087 0.00257239 0.00262026 0.00261293 0.00257622
 0.00257269 0.00268261]


In [ ]:
from sklearn.metrics import roc_auc_score, average_precision_score, brier_score_loss

print("=== Before Calibration ===")
print("ROC-AUC:", roc_auc_score(y_test, y_prob))
print("PR-AUC:", average_precision_score(y_test, y_prob))
print("Brier Score:", brier_score_loss(y_test, y_prob))

print("\n=== After Calibration ===")
print("ROC-AUC:", roc_auc_score(y_test, y_prob_calibrated))
print("PR-AUC:", average_precision_score(y_test, y_prob_calibrated))
print("Brier Score:", brier_score_loss(y_test, y_prob_calibrated))

=== Before Calibration ===
ROC-AUC: 0.9618533403699644
PR-AUC: 0.5971084543152967
Brier Score: 0.03252205

=== After Calibration ===
ROC-AUC: 0.9596882231153331
PR-AUC: 0.593154239196992
Brier Score: 0.03454249159084572


In [ ]:
import numpy as np

print("Minimum:", y_prob.min())
print("Maximum:", y_prob.max())
print("Mean:", y_prob.mean())

print("\nPercentiles:")
for p in [25, 50, 75, 90, 95, 99]:
    print(f"{p}%:", np.percentile(y_prob, p))

Minimum: 0.0
Maximum: 1.0
Mean: 0.08011

Percentiles:
25%: 0.0
50%: 0.005
75%: 0.045
90%: 0.29
95%: 0.5652499999999997
99%: 0.92


In [ ]:
def assess_risk(probability):
    percentage = probability * 100

    if percentage < 30:
        risk = "Low"
        recommendation = "Normal operation"
    elif percentage < 50:
        risk = "Medium"
        recommendation = "Inspection recommended"
    elif percentage < 70:
        risk = "High"
        recommendation = "Inspect the machine soon"
    else:
        risk = "Very High"
        recommendation = "Immediate inspection recommended"

    return round(percentage, 2), risk, recommendation


# مثال
probability = y_prob[9]

percentage, risk, recommendation = assess_risk(probability)

print("Failure Probability:", percentage, "%")
print("Risk Level:", risk)
print("Recommendation:", recommendation)

Failure Probability: 34.0 %
Risk Level: Medium
Recommendation: Inspection recommended


المودل الاول كملناه الذي يحدد خطوره الاله

In [ ]:
# تجهيز بيانات Model 2

df_model2 = df[
    (df["Machine failure"] == 1) &
    (df["Failure Type"] != "No Failure")
].copy()

# تحويل Multiple Failure إلى Other
df_model2["Failure Type"] = df_model2["Failure Type"].replace(
    "Multiple Failure",
    "Other"
)

print("عدد الصفوف:", len(df_model2))
print("\nتوزيع أنواع الأعطال:")
print(df_model2["Failure Type"].value_counts())

عدد الصفوف: 330

توزيع أنواع الأعطال:
Failure Type
HDF      106
PWF       80
OSF       78
TWF       42
Other     24
Name: count, dtype: int64


In [ ]:
from sklearn.model_selection import train_test_split

features_m2 = [
    "Type",
    "Air temperature [K]",
    "Process temperature [K]",
    "Rotational speed [rpm]",
    "Torque [Nm]",
    "Tool wear [min]"
]

target_m2 = "Failure Type"

X2 = df_model2[features_m2].copy()
y2 = df_model2[target_m2].copy()

# 70% Train - 15% Validation - 15% Test
X2_train, X2_temp, y2_train, y2_temp = train_test_split(
    X2,
    y2,
    test_size=0.30,
    random_state=42,
    stratify=y2
)

X2_val, X2_test, y2_val, y2_test = train_test_split(
    X2_temp,
    y2_temp,
    test_size=0.50,
    random_state=42,
    stratify=y2_temp
)

print("Train:", X2_train.shape)
print(y2_train.value_counts())

print("\nValidation:", X2_val.shape)
print(y2_val.value_counts())

print("\nTest:", X2_test.shape)
print(y2_test.value_counts())

Train: (231, 6)
Failure Type
HDF      74
PWF      56
OSF      55
TWF      29
Other    17
Name: count, dtype: int64

Validation: (49, 6)
Failure Type
HDF      16
PWF      12
OSF      11
TWF       6
Other     4
Name: count, dtype: int64

Test: (50, 6)
Failure Type
HDF      16
PWF      12
OSF      12
TWF       7
Other     3
Name: count, dtype: int64


In [ ]:
from sklearn.preprocessing import LabelEncoder

encoder_m2 = LabelEncoder()

# تدريب الـ Encoder على Train فقط
X2_train = X2_train.copy()
X2_val = X2_val.copy()
X2_test = X2_test.copy()

X2_train["Type"] = encoder_m2.fit_transform(X2_train["Type"])

X2_val["Type"] = encoder_m2.transform(X2_val["Type"])
X2_test["Type"] = encoder_m2.transform(X2_test["Type"])

print("Type Mapping:")
for label, value in zip(encoder_m2.classes_, encoder_m2.transform(encoder_m2.classes_)):
    print(label, "=", value)

print("\nTrain sample:")
print(X2_train.head())

print("\nData types:")
print(X2_train.dtypes)

Type Mapping:
H = 0
L = 1
M = 2

Train sample:
      Type  Air temperature [K]  Process temperature [K]  \
9974     1                298.6                    308.2   
9664     1                299.1                    310.2   
4620     1                303.1                    311.3   
161      1                298.3                    308.1   
3611     1                301.7                    310.9   

      Rotational speed [rpm]  Torque [Nm]  Tool wear [min]  
9974                    1361         68.2              172  
9664                    1317         54.8              231  
4620                    1336         52.6              172  
161                     1412         52.3              218  
3611                    1405         46.4              207  

Data types:
Type                         int64
Air temperature [K]        float64
Process temperature [K]    float64
Rotational speed [rpm]       int64
Torque [Nm]                float64
Tool wear [min]              int64
dty

In [ ]:
from imblearn.over_sampling import SMOTENC
import pandas as pd
import numpy as np

# تحويل البيانات إلى NumPy
X_train_np = X2_train.values
y_train_np = y2_train.values

# Type هو العمود رقم 0 وهو categorical
smote = SMOTENC(
    categorical_features=[0],
    random_state=42
)

X_smote, y_smote = smote.fit_resample(
    X_train_np,
    y_train_np
)

# تحويل إلى DataFrame
X_smote = pd.DataFrame(
    X_smote,
    columns=features_m2
)

y_smote = pd.Series(y_smote, name=target_m2)

print("بعد SMOTENC:")
print(y_smote.value_counts())
print("Shape:", X_smote.shape)

بعد SMOTENC:
Failure Type
Other    74
OSF      74
HDF      74
PWF      74
TWF      74
Name: count, dtype: int64
Shape: (370, 6)


In [ ]:
# عدد العينات المطلوب
TARGET_SIZE = 5000

# نحدد عدد العينات لكل فئة
classes = y_smote.unique()
samples_per_class = TARGET_SIZE // len(classes)

X_final = []
y_final = []

np.random.seed(42)

for cls in classes:

    # بيانات الفئة
    X_cls = X_smote[y_smote == cls].copy()

    # سحب عينات مع التكرار
    X_sample = X_cls.sample(
        n=samples_per_class,
        replace=True,
        random_state=42
    ).reset_index(drop=True)

    # إضافة Jittering للأعمدة الرقمية فقط
    numeric_cols = features_m2[1:]

    for col in numeric_cols:
        std = X_cls[col].std()
        noise = np.random.normal(
            0,
            std * 0.02,
            samples_per_class
        )
        X_sample[col] = X_sample[col] + noise

    X_final.append(X_sample)
    y_final.extend([cls] * samples_per_class)

# دمج جميع الفئات
X2_train_final = pd.concat(
    X_final,
    ignore_index=True
)

y2_train_final = pd.Series(
    y_final,
    name=target_m2
)

print("Final Training Shape:", X2_train_final.shape)

print("\nClass Distribution:")
print(y2_train_final.value_counts())

print("\nMissing Values:")
print(X2_train_final.isnull().sum())

Final Training Shape: (5000, 6)

Class Distribution:
Failure Type
Other    1000
OSF      1000
HDF      1000
PWF      1000
TWF      1000
Name: count, dtype: int64

Missing Values:
Type                       0
Air temperature [K]        0
Process temperature [K]    0
Rotational speed [rpm]     0
Torque [Nm]                0
Tool wear [min]            0
dtype: int64


In [ ]:
from sklearn.ensemble import RandomForestClassifier

model2 = RandomForestClassifier(
    n_estimators=500,
    random_state=42,
    n_jobs=-1
)

model2.fit(
    X2_train_final,
    y2_train_final
)

print("Model 2 trained successfully!")

Model 2 trained successfully!


In [ ]:
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix
)

# التنبؤ على Validation الأصلية
y2_val_pred = model2.predict(X2_val)

# Accuracy الكلية
accuracy_m2 = accuracy_score(y2_val, y2_val_pred)

print("Model 2 Accuracy:", round(accuracy_m2 * 100, 2), "%")

print("\nClassification Report:")
print(classification_report(y2_val, y2_val_pred))

print("\nConfusion Matrix:")
print(confusion_matrix(y2_val, y2_val_pred))

Model 2 Accuracy: 85.71 %

Classification Report:
              precision    recall  f1-score   support

         HDF       0.94      0.94      0.94        16
         OSF       0.82      0.82      0.82        11
       Other       0.50      0.50      0.50         4
         PWF       0.91      0.83      0.87        12
         TWF       0.86      1.00      0.92         6

    accuracy                           0.86        49
   macro avg       0.80      0.82      0.81        49
weighted avg       0.86      0.86      0.86        49


Confusion Matrix:
[[15  0  0  1  0]
 [ 0  9  1  0  1]
 [ 0  2  2  0  0]
 [ 1  0  1 10  0]
 [ 0  0  0  0  6]]


In [ ]:
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix
)

# التنبؤ على Test الأصلية
y2_test_pred = model2.predict(X2_test)

# Accuracy
accuracy_test = accuracy_score(
    y2_test,
    y2_test_pred
)

print("Model 2 Test Accuracy:",
      round(accuracy_test * 100, 2), "%")

print("\nClassification Report:")
print(classification_report(
    y2_test,
    y2_test_pred
))

print("\nConfusion Matrix:")
print(confusion_matrix(
    y2_test,
    y2_test_pred
))

Model 2 Test Accuracy: 88.0 %

Classification Report:
              precision    recall  f1-score   support

         HDF       1.00      1.00      1.00        16
         OSF       0.82      0.75      0.78        12
       Other       0.25      0.33      0.29         3
         PWF       0.92      1.00      0.96        12
         TWF       1.00      0.86      0.92         7

    accuracy                           0.88        50
   macro avg       0.80      0.79      0.79        50
weighted avg       0.89      0.88      0.88        50


Confusion Matrix:
[[16  0  0  0  0]
 [ 0  9  3  0  0]
 [ 0  2  1  0  0]
 [ 0  0  0 12  0]
 [ 0  0  0  1  6]]


In [ ]:
from google.colab import drive
import os
import joblib

# ربط Google Drive
drive.mount('/content/drive')

# مجلد المشروع
project_path = "/content/drive/MyDrive/مشروع التنبؤ بالاعطال"
os.makedirs(project_path, exist_ok=True)

# حفظ Model 1
joblib.dump(
    model1,
    f"{project_path}/model1_failure_prediction.pkl"
)

# حفظ Model 2
joblib.dump(
    model2,
    f"{project_path}/model2_failure_type.pkl"
)

# حفظ Encoder الخاص بـ Model 1
joblib.dump(
    encoder,
    f"{project_path}/model1_type_encoder.pkl"
)

# حفظ Encoder الخاص بـ Model 2
joblib.dump(
    encoder_m2,
    f"{project_path}/model2_type_encoder.pkl"
)

print("تم الحفظ بنجاح ✅")
print("\nالمجلد:")
print(project_path)

print("\nالملفات:")
for file in os.listdir(project_path):
    print("-", file)

Mounted at /content/drive
تم الحفظ بنجاح ✅

المجلد:
/content/drive/MyDrive/مشروع التنبؤ بالاعطال

الملفات:
- model1_failure_prediction.pkl
- model2_failure_type.pkl
- model1_type_encoder.pkl
- model2_type_encoder.pkl


In [ ]:
# ============================================================
# دمج Model 1 + Model 2 + Decoder
# مشروع التنبؤ بالاعطال
# ============================================================

def predict_machine(
    machine_type,
    air_temperature,
    process_temperature,
    rotational_speed,
    torque,
    tool_wear
):

    # --------------------------------------------------------
    # 1) ترميز Type
    # --------------------------------------------------------

    encoded_type = encoder.transform([machine_type])[0]

    data = [[
        encoded_type,
        air_temperature,
        process_temperature,
        rotational_speed,
        torque,
        tool_wear
    ]]

    # --------------------------------------------------------
    # 2) Model 1 - احتمال العطل
    # --------------------------------------------------------

    probability = model1.predict_proba(data)[0][1]

    percentage = round(probability * 100, 2)

    # --------------------------------------------------------
    # 3) تحديد مستوى الخطورة
    # --------------------------------------------------------

    if percentage < 30:
        risk = "Low"
        recommendation = "Normal operation"

    elif percentage < 50:
        risk = "Medium"
        recommendation = "Inspection recommended"

    elif percentage < 70:
        risk = "High"
        recommendation = "Inspect the machine soon"

    else:
        risk = "Very High"
        recommendation = "Immediate inspection recommended"

    # --------------------------------------------------------
    # 4) Model 2 - نوع العطل
    # --------------------------------------------------------

    if percentage >= 30:

        predicted_type = model2.predict(data)[0]

        # Decoder
        failure_decoder = {
            "HDF": "High Heat Dissipation Failure",
            "PWF": "Power Failure",
            "OSF": "Overstrain Failure",
            "TWF": "Tool Wear Failure",
            "Other": "Other"
        }

        failure_type = failure_decoder.get(
            predicted_type,
            "Other"
        )

    else:
        failure_type = "No Failure"

    # --------------------------------------------------------
    # 5) النتيجة النهائية
    # --------------------------------------------------------

    return {
        "Failure Probability": percentage,
        "Risk Level": risk,
        "Recommendation": recommendation,
        "Failure Type": failure_type
    }


print("تم إنشاء دالة الدمج والـ Decoder بنجاح ✅")

تم إنشاء دالة الدمج والـ Decoder بنجاح ✅


In [ ]:
def predict_machine(
    machine_type,
    air_temperature,
    process_temperature,
    rotational_speed,
    torque,
    tool_wear
):

    # إنشاء DataFrame بنفس أعمدة التدريب
    data = pd.DataFrame([[
        machine_type,
        air_temperature,
        process_temperature,
        rotational_speed,
        torque,
        tool_wear
    ]], columns=features)

    # ترميز Type
    data["Type"] = encoder.transform(data["Type"])

    # Model 1
    probability = model1.predict_proba(data)[0][1]

    percentage = round(probability * 100, 2)

    # Risk Level
    if percentage < 30:
        risk = "Low"
        recommendation = "Normal operation"

    elif percentage < 50:
        risk = "Medium"
        recommendation = "Inspection recommended"

    elif percentage < 70:
        risk = "High"
        recommendation = "Inspect the machine soon"

    else:
        risk = "Very High"
        recommendation = "Immediate inspection recommended"

    # Model 2
    if percentage >= 30:

        predicted_type = model2.predict(data)[0]

        failure_decoder = {
            "HDF": "High Heat Dissipation Failure",
            "PWF": "Power Failure",
            "OSF": "Overstrain Failure",
            "TWF": "Tool Wear Failure",
            "Other": "Other"
        }

        failure_type = failure_decoder.get(
            predicted_type,
            "Other"
        )

    else:
        failure_type = "No Failure"

    return {
        "Failure Probability": percentage,
        "Risk Level": risk,
        "Recommendation": recommendation,
        "Failure Type": failure_type
    }


print("تم تحديث دالة الدمج وإزالة التحذير ✅")

تم تحديث دالة الدمج وإزالة التحذير ✅


In [ ]:
# حفظ إعدادات ودالة مشروع التنبؤ بالأعطال

import joblib
import os

project_path = "/content/drive/MyDrive/مشروع التنبؤ بالاعطال"

# حفظ أسماء الأعمدة والإعدادات
project_config = {
    "features": features,
    "features_m2": features_m2,
    "target_model1": target,
    "target_model2": target_m2,

    "risk_thresholds": {
        "Low": "< 30%",
        "Medium": "30% - <50%",
        "High": "50% - <70%",
        "Very High": ">= 70%"
    },

    "failure_types": [
        "HDF",
        "OSF",
        "PWF",
        "TWF",
        "Other"
    ]
}

joblib.dump(
    project_config,
    f"{project_path}/project_config.pkl"
)

# حفظ دالة التنبؤ
joblib.dump(
    predict_machine,
    f"{project_path}/predict_machine.pkl"
)

print("تم حفظ إعدادات المشروع ودالة التنبؤ ✅")

print("\nملفات المشروع:")
for file in os.listdir(project_path):
    print("-", file)

تم حفظ إعدادات المشروع ودالة التنبؤ ✅

ملفات المشروع:
- model1_failure_prediction.pkl
- model2_failure_type.pkl
- model1_type_encoder.pkl
- model2_type_encoder.pkl
- project_config.pkl
- predict_machine.pkl


In [ ]:
# ============================================================
# مشروع التنبؤ بالاعطال
# الواجهة العربية + ربط Model 1 و Model 2
# ============================================================

!pip -q install gradio

import gradio as gr
import joblib
import pandas as pd

# ============================================================
# 1) تحميل المودلات من Google Drive
# ============================================================

project_path = "/content/drive/MyDrive/مشروع التنبؤ بالاعطال"

model1 = joblib.load(
    f"{project_path}/model1_failure_prediction.pkl"
)

model2 = joblib.load(
    f"{project_path}/model2_failure_type.pkl"
)

encoder = joblib.load(
    f"{project_path}/model1_type_encoder.pkl"
)

encoder_m2 = joblib.load(
    f"{project_path}/model2_type_encoder.pkl"
)

print("تم تحميل المودلات بنجاح ✅")


# ============================================================
# 2) دالة التنبؤ
# ============================================================

def predict_machine(
    machine_type,
    air_temperature,
    process_temperature,
    rotational_speed,
    torque,
    tool_wear
):

    # التأكد من إدخال جميع البيانات
    values = [
        machine_type,
        air_temperature,
        process_temperature,
        rotational_speed,
        torque,
        tool_wear
    ]

    if any(v is None or v == "" for v in values):
        return (
            "⚠️ يرجى إدخال جميع بيانات الآلة.",
            "",
            "",
            ""
        )

    # إنشاء DataFrame بنفس ترتيب التدريب
    data = pd.DataFrame([[
        machine_type,
        air_temperature,
        process_temperature,
        rotational_speed,
        torque,
        tool_wear
    ]], columns=[
        "Type",
        "Air temperature [K]",
        "Process temperature [K]",
        "Rotational speed [rpm]",
        "Torque [Nm]",
        "Tool wear [min]"
    ])

    # ترميز نوع الآلة
    data["Type"] = encoder.transform(data["Type"])

    # ========================================================
    # Model 1
    # ========================================================

    probability = model1.predict_proba(data)[0][1]
    percentage = round(probability * 100, 2)

    # ========================================================
    # مستوى الخطورة
    # ========================================================

    if percentage < 30:
        risk = "منخفض"
        recommendation = "التشغيل طبيعي ولا توجد حاجة لفحص فوري."

    elif percentage < 50:
        risk = "متوسط"
        recommendation = "يُنصح بفحص الآلة."

    elif percentage < 70:
        risk = "مرتفع"
        recommendation = "يُنصح بفحص الآلة في أقرب وقت."

    else:
        risk = "مرتفع جدًا"
        recommendation = "يوصى بإيقاف الآلة وفحصها بشكل فوري."

    # ========================================================
    # Model 2
    # ========================================================

    if percentage >= 30:

        predicted_type = model2.predict(data)[0]

        decoder = {
            "HDF": "عطل تبديد الحرارة",
            "PWF": "عطل الطاقة",
            "OSF": "عطل الإجهاد الزائد",
            "TWF": "عطل تآكل الأداة",
            "Other": "أخرى"
        }

        failure_type = decoder.get(
            predicted_type,
            "أخرى"
        )

    else:
        failure_type = "لا يوجد عطل متوقع"

    # ========================================================
    # النتائج
    # ========================================================

    probability_result = f"{percentage}%"

    risk_result = risk

    recommendation_result = recommendation

    failure_result = failure_type

    return (
        probability_result,
        risk_result,
        recommendation_result,
        failure_result
    )


# ============================================================
# 3) CSS — التصميم
# ============================================================

css = """

body {
    direction: rtl;
}

.gradio-container {
    max-width: 1100px !important;
    margin: auto !important;
    font-family: Arial, Tahoma, sans-serif;
}

/* العنوان */

.title-box {
    text-align: center;
    padding: 25px;
    border-radius: 20px;
    margin-bottom: 20px;
    border: 2px solid #555;
}

.main-title {
    font-size: 42px;
    font-weight: 900;
    margin-bottom: 10px;
}

.offer {
    font-size: 20px;
    font-weight: bold;
}

/* الأقسام */

.section-title {
    font-size: 25px;
    font-weight: 800;
    margin-top: 10px;
    margin-bottom: 15px;
}

/* زر التنبؤ */

.predict-btn {
    font-size: 22px !important;
    font-weight: 900 !important;
    height: 60px !important;
    border-radius: 15px !important;
}

/* النتائج */

.result-box {
    text-align: center;
    border-radius: 18px;
    padding: 18px;
    border: 2px solid #555;
}

.result-value {
    font-size: 28px;
    font-weight: 900;
}

/* جؤ */

.joo {
    text-align: center;
    font-size: 55px;
    font-weight: 1000;
    margin-top: 35px;
    letter-spacing: 8px;
}

/* التنبيه */

.warning {
    text-align: center;
    font-size: 15px;
    font-weight: bold;
    margin-top: 20px;
    padding: 15px;
    border-radius: 12px;
    border: 1px solid #777;
}

"""

# ============================================================
# 4) بناء الواجهة
# ============================================================

with gr.Blocks(
    css=css,
    title="مشروع التنبؤ بالاعطال"
) as app:

    gr.HTML("""
    <div class="title-box">

        <div class="main-title">
            🔩 انا عم الكل
        </div>

        <div class="offer">
            إجابات أفضل ودقة أقوى — اشترك بالعرض بـ 50 دولار فقط
        </div>

    </div>
    """)

    gr.Markdown(
        "## ⚙️ أدخل بيانات الآلة",
        elem_classes="section-title"
    )

    with gr.Row():

        machine_type = gr.Dropdown(
            choices=["H", "M", "L"],
            label="نوع الآلة",
            info="اختر نوع الآلة",
            value=None
        )

        air_temperature = gr.Number(
            label="درجة حرارة الهواء",
            info="الوحدة: K",
            value=None
        )

        process_temperature = gr.Number(
            label="درجة حرارة العملية",
            info="الوحدة: K",
            value=None
        )

    with gr.Row():

        rotational_speed = gr.Number(
            label="سرعة الدوران",
            info="الوحدة: rpm",
            value=None
        )

        torque = gr.Number(
            label="عزم الدوران",
            info="الوحدة: Nm",
            value=None
        )

        tool_wear = gr.Number(
            label="تآكل الأداة",
            info="الوحدة: min",
            value=None
        )

    predict_btn = gr.Button(
        "🔩 تحليل الآلة والتنبؤ بالعطل",
        variant="primary",
        elem_classes="predict-btn"
    )

    gr.Markdown(
        "## 📊 نتيجة التحليل",
        elem_classes="section-title"
    )

    with gr.Row():

        probability_output = gr.Textbox(
            label="احتمال حدوث العطل",
            interactive=False,
            elem_classes="result-box"
        )

        risk_output = gr.Textbox(
            label="مستوى الخطورة",
            interactive=False,
            elem_classes="result-box"
        )

    with gr.Row():

        recommendation_output = gr.Textbox(
            label="التوصية",
            interactive=False,
            elem_classes="result-box"
        )

        failure_output = gr.Textbox(
            label="نوع العطل المتوقع",
            interactive=False,
            elem_classes="result-box"
        )

    predict_btn.click(
        fn=predict_machine,
        inputs=[
            machine_type,
            air_temperature,
            process_temperature,
            rotational_speed,
            torque,
            tool_wear
        ],
        outputs=[
            probability_output,
            risk_output,
            recommendation_output,
            failure_output
        ]
    )

    gr.HTML("""
    <div class="joo">
        🔩 جؤ 🔩
    </div>

    <div class="warning">
        ⚠️ تنبيه: النموذج معرض للخطأ، لذا لا تعتمد على نتائجه بنسبة 100%.
    </div>
    """)


# ============================================================
# 5) تشغيل الواجهة
# ============================================================

app.launch(share=True)

تم تحميل المودلات بنجاح ✅


/tmp/ipykernel_720/2235883656.py:256: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: css. Please pass these parameters to launch() instead.
  with gr.Blocks(


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://0da402118b5d9ec395.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
